In [4]:
import cv2
import numpy as np
import torch

# GLOBAL MEAN & STD computed from your OLD HDF5 dataset
MEAN = -2.6647588e-06
STD  = 0.9996273

def preprocess_image(path):
    # 1. Read
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # 2. Resize
    img = cv2.resize(img, (224, 224))

    # 3. Denoise
    img = cv2.fastNlMeansDenoisingColored(img, None, 10, 10, 7, 21)

    # 4. Histogram Equalization (Y channel)
    img_yuv = cv2.cvtColor(img, cv2.COLOR_RGB2YUV)
    img_yuv[:, :, 0] = cv2.equalizeHist(img_yuv[:, :, 0])
    img = cv2.cvtColor(img_yuv, cv2.COLOR_YUV2RGB)

    # 5. Sharpen
    kernel = np.array([
        [0, -1, 0],
        [-1, 5, -1],
        [0, -1, 0]
    ])
    img = cv2.filter2D(img, -1, kernel)

    # 6. Convert to float
    img = img.astype(np.float32)

    # 7. Normalize 0–1
    img = img / 255.0

    # 8. Standardize using OLD dataset stats
    img = (img - MEAN) / (STD + 1e-7)

    # 9. CHW format
    img = torch.tensor(img).permute(2, 0, 1).float()

    # 10. Add batch dimension
    return img.unsqueeze(0)
